<a href="https://colab.research.google.com/github/jensjose07/deep_learning_lab/blob/main/expeiment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tensorflow as tf
import time
import pandas as pd

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.utils import to_categorical


# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize input images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encode class labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

epochs = 5
batch_size = 64


# Build feed-forward neural network
def build_model(initializer="glorot_uniform",
                dropout=False,
                regularizer=None):

    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),

        Dense(
            512,
            activation="relu",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer
        ),

        Dropout(0.5) if dropout else tf.keras.layers.Activation("linear"),

        Dense(
            256,
            activation="relu",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer
        ),

        Dropout(0.5) if dropout else tf.keras.layers.Activation("linear"),

        Dense(
            64,
            activation="relu",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer
        ),

        Dropout(0.5) if dropout else tf.keras.layers.Activation("linear"),

        Dense(
            10,
            activation="softmax",
            kernel_initializer=initializer
        )
    ])

    return model


# Define model configurations
models = {
    "Baseline": {
        "initializer": "glorot_uniform",
        "dropout": False,
        "regularizer": None,
        "regularization_name": "None"
    },

    "Xavier": {
        "initializer": "glorot_uniform",
        "dropout": False,
        "regularizer": None,
        "regularization_name": "None"
    },

    "Kaiming": {
        "initializer": "he_normal",
        "dropout": False,
        "regularizer": None,
        "regularization_name": "None"
    },

    "Dropout": {
        "initializer": "glorot_uniform",
        "dropout": True,
        "regularizer": None,
        "regularization_name": "Dropout"
    },

    "L1 Regularization": {
        "initializer": "glorot_uniform",
        "dropout": False,
        "regularizer": l1(0.001),
        "regularization_name": "L1"
    },

    "L2 Regularization": {
        "initializer": "glorot_uniform",
        "dropout": False,
        "regularizer": l2(0.001),
        "regularization_name": "L2"
    }
}


# Optimizers
optimizers = {
    "Adam": lambda: tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    "SGD": lambda: tf.keras.optimizers.SGD(
        learning_rate=0.01
    )
}


# Store results
results = []


# Train every model with every optimizer
for model_name, config in models.items():

    for optimizer_name, optimizer_fn in optimizers.items():

        model = build_model(
            initializer=config["initializer"],
            dropout=config["dropout"],
            regularizer=config["regularizer"]
        )

        optimizer = optimizer_fn()

        model.compile(
            optimizer=optimizer,
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        start_time = time.time()

        model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(x_test, y_test),
            verbose=0
        )

        training_time = time.time() - start_time

        test_loss, test_accuracy = model.evaluate(
            x_test,
            y_test,
            verbose=0
        )

        # Store results including Regularization
        results.append([
            model_name,
            optimizer_name,
            config["regularization_name"],
            test_loss,
            test_accuracy * 100,
            training_time
        ])


# Convert results into DataFrame
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Optimizer",
        "Regularization",
        "Test Loss",
        "Test Accuracy (%)",
        "Training Time (seconds)"
    ]
)


# Display final comparison
print("\n" + "=" * 100)
print("FINAL COMPARISON")
print("=" * 100)

print(results_df.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



FINAL COMPARISON
            Model Optimizer Regularization  Test Loss  Test Accuracy (%)  Training Time (seconds)
         Baseline      Adam           None   1.488834          47.049999                23.822026
         Baseline       SGD           None   1.600381          41.659999                18.060899
           Xavier      Adam           None   1.518777          45.809999                19.124200
           Xavier       SGD           None   1.708676          38.350001                17.558999
          Kaiming      Adam           None   1.508914          46.140000                19.749539
          Kaiming       SGD           None   1.623469          42.539999                19.122231
          Dropout      Adam        Dropout   1.825351          35.190001                24.712048
          Dropout       SGD        Dropout   1.880527          33.539999                20.453542
L1 Regularization      Adam             L1   2.385286          32.589999                20.435012
L1